In [ ]:
"""
<분류분석에서 사용되는 모델>
- 옛날 모델 : KNN, LogisticRegression(로지스틱레그레이션, 다중분류모델)

- 그다음 향상 모델 : DecisionTree(DT, 결정트리, 의사결정나무라고 칭함, 트리구조 1개 사용)

- 그다음 향상 모델(최신, 트리구조 1개 이상 사용)
  : RandomForest, ExtraTrees

- 그다음 향상 모델(성능 향상 최신, 훈련 중 데이터 섞기 방법 및 속도 개선)
  : GradientBoosting, HistGradientBoosting, XGBoost, LightGBM, 기타 등등

 - 상관관계 검정부터 분류 평가까지 모두 진행...
"""

In [ ]:
### 사용할 라이브러리 불러들이기
# 데이터 조작을 위한 라이브러리
import pandas as pd

# 기본 시각화 라이브러리
import matplotlib.pyplot as plt

# 통계적 시각화 라이브러리(보통 기본 시각화보다 향상된 라이브러라고 생각하면 됨)
import seaborn as sns

# 시각화 시 그래프의 너비/높이를 기본 설정하기(단위=인치)
plt.rcParams["figure.figsize"] = (12, 5)

# 시각화시 한글 깨짐 처리를 위한 폰트 지정(Windows OS 기준)
plt.rcParams["font.family"] = "Malgun Gothic"

# 시각화시 마이너스(-) 기호 깨짐 방지 처리
plt.rcParams["axes.unicode_minus"] = False

### 상관관계 검정(scipy : 통계 라이브러리)
from scipy.stats import spearmanr, pearsonr

### 훈련/검증/테스트 데이터로 분리하는 함수
from sklearn.model_selection import train_test_split

### 데이터스케일링 표준화 클래스
from sklearn.preprocessing import StandardScaler

### 사용할 훈련모델 클래스
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

### 분류분석 오차확인(평가) 라이브러리
# - 오차행렬표, 오차행렬도(그래프)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

### 분류모델 평가지표(선정기준)
# - 정확도, 정밀도, 재현율, f1-score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
### 사용할 데이터 : 08_wine.csv
# 사용할 변수명 : wine
# 데이터 기본정보 및 기초통계 확인
file_path = '../data/08_wine.csv'
wine = pd.read_csv(file_path)
wine.head(3)

In [ ]:
### 상관관계 계수 확인하기
# - 상관관계 확인시에는 독립변수 및 종속변수로 나누지 않고 합니다.
# - 상관관계 검증시에는 독립변수와 종속변수로 구분해야함


### <상관관계 분석 및 검증 순서>
# 1. 상관관계 메트릭스표 확인(상관관계 계수 확인)
# 2. 상관관계도 시각화(히트맵-상관관계 계수를 색으로 확인)
# 3. 상관행렬도 시각화(산점도 분포 확인-선형/비선형 형태 확인)
# 4. 상관관계 검정
#    : 종속변수와 독립변수 각각에 대한 유의미성 확인
#    : 독립과 종속변수 각각 구분하여 진행
#    : 검정방법 (스피어만, 피어슨 방법 사용)

In [ ]:
# 1. 상관관계 메트릭스표 확인(상관관계 계수 확인)
corr = wine.corr()
corr

In [ ]:
# 2. 상관관계도 시각화(히트맵-상관관계 계수를 색으로 확인)
sns.heatmap(corr, annot=True,cmap="Oranges", fmt='.2f')
plt.show()

In [ ]:
sns.pairplot(wine)
plt.show()

In [ ]:
# - 4. Correlation & p-value

data = wine[["alcohol", "pH", "sugar"]]
target = wine["class"]
data.shape, target.shape

### 상관관계 검정 : 피어슨 방식 (선형 가능)
for column in data.columns :
    s, p = pearsonr(data[column], target)

    if p < 0.05 :
        print(f"{column} vs class 피어슨 상관관계 검정 p-value는 {p:.4f}로 p-value < 0.05, 유의미함")
    else :
        print(f"{column} vs class 피어슨 상관관계 검정 p-value는 {p:.4f}로 p-value >= 0.05, 유의미 하지않음")

### 상관관계 검정 : 스피어만 방식 (선형 가능)
for column in data.columns :
    s, p = spearmanr(data[column], target)

    if p < 0.05 :
        print(f"{column} vs class 스피어만 상관관계 검정 p-value는 {p:.4f}로 p-value < 0.05, 유의미함")
    else :
        print(f"{column} vs class 스피어만 상관관계 검정 p-value는 {p:.4f}로 p-value >= 0.05, 유의미 하지않음")


# pr_alcohol = pearsonr(data["alcohol"], target)
# pr_pH = pearsonr(data["pH"], target)
# pr_sugar = pearsonr(data["sugar"], target)
# pr = [pr_alcohol, pr_pH, pr_sugar]
# display(pr)

# sp_alcohol = spearmanr(data["alcohol"], target)
# sp_pH = spearmanr(data["pH"], target)
# sp_sugar = spearmanr(data["sugar"], target)
# sp = [sp_alcohol, sp_pH, sp_sugar]
# display(sp)

# Pearson R의 p-value가 0.05보다 작으므로 Pearson R 사용 


In [ ]:
### (종합 해석)
# - 상관관계 계수의 경우 알콜의 계수값은 0.03으로 매우 낮은 상관성을 띄고 있으며,
#   -> 당도 및 농도의 경우에는 0.3 이상으로 낮지만 상관성을 띄고 있음
#   -> 알콜의 경우 향후 모델 훈련 시 제외하고 진행하여,
#      성능에 영향을 미치는지도 확인할 필요성이 있음


# - 전체 특성들의 분포를 볼 때,
#   -> 와인종류의 0과 1을 기준으로 수평 분포를 나타내고 있기에
#   -> 분류 분석으로 진행하는 것이 타당함


### (해석)
# - 실제 문서에 작성할 때는 한가지 방법으로만 작성
# - 하지만, 분석가 입장에서는 모두다 확인 후 최종 훈련이 완료된 다음에 결정


### 선형/비선형 모두 가능한 검정방식인 스피어만 검정 방법에서는
#   - 알콜의 경우 p-value >= 0.05 이므로, 유의미하지 않았으며,
#   - 당도 및 농도의 경우 p-value < 0.05를 만족하여 유의미한 것으로 확인됨


### 선형을 주로 검정하는 피어슨 검정 방법에서는
#   - 모든 특성들이 p-value < 0.05를 만족하여 유의미하므로, 모두 훈련에 사용가능


### 따라서, 이론상으로는 스피어만 방식을 준수해야하나,
#   모델 훈련 시에는 모든 특성을 적용하여 훈련 후,
#   유의미하지 않은 알콜을 제외하고, 재훈련하여 비교할 필요성은 있음


In [ ]:
print(type(data))
print(type(target))

data.to_numpy()
target.to_numpy()

In [ ]:
train_input, test_input, train_target, test_target = train_test_split(data, target, test_size= 0.2, stratify=target, random_state= 42)

print(train_input.shape, test_input.shape)
print(train_target.shape, test_target.shape)

In [ ]:
ss = StandardScaler()

ss.fit(train_input)
train_scaled = ss.transform(train_input)
test_scaled = ss.transform(test_input)

print(train_scaled.shape, train_target.shape)
print(test_scaled.shape, test_target.shape)

knn = KNeighborsClassifier(n_neighbors= 5)
knn.fit(train_scaled, train_target)

train_r2 = knn.score(train_scaled, train_target)
test_r2 = knn.score(test_scaled, test_target)

print(train_r2, test_r2, train_r2 - test_r2)


In [ ]:
"""
<로지스틱 레그레이션 모델>
 - 이름은 회귀이지만, 분류분석에 사용되는 모델 입니다.
 - 내부적으로 사용되는 산술식이 "다중회귀" 모델의 알고리즘을 사용
 - 이진분류, 다중분류에 사용됩니다.
 
<이진분류 vs 다중분류>
 - 종속변수의 범주의 갯수에 따라 이진 또는 다중 분류로 구분합니다.
 - 이진분류 : 범주가 2개인 경우 (둘중 하나 선택, 남자/여자, 0/1)
 - 다중분류 : 범주가 3개이상인 경우(여러개 중 하나 선택, 0~n)


<로지스틱레그레이션 모델 성능 향상을 위한 하이퍼파라메터 튜닝>
 - 튜닝에 사용되는 기본적인 변수명 : C(규제강도), max_iter(훈련 반복횟수)
 - C(규제강도)
   : 규제강도는 릿지 또는 라쏘에서 사용한 alpha 규제강도와 동일한 개념으로
     복잡도를 높이거나 낮추는 방식을 적용함
   : C의 기본값은 = 1
   : C 값의 범위 : 0.001~100 사이를 주로 사용
   : C의 값이 커질수록 규제를 완화시키며, 작을수록 규제를 강화시킴
   
 - max_iter(훈련 반복횟수)
   : 훈련을 여러번 반복하여 훈련의 정확도를 높임
   : max_iter의 기본값은 = 100
   : max_iter의 범위 : 1이상~
   
***(중요) 하이퍼파라메터 튜닝을 한다고 해서 꼭~ 모델의 성능이 좋아진다고 볼수는 없음
          오히려 나빠지는 경우도 가끔 발생함
"""

In [ ]:
###로지스틱레그레이션으로 훈련 후 과적합 여부까지 진행
# - 규제강도 = 0.13, 반복횟수 = 100회
lr = LogisticRegression(C=0.13, max_iter=100)

lr.fit(train_scaled, train_target)

train_score = round(lr.score(train_scaled, train_target), 4)
test_score = round(lr.score(test_scaled, test_target), 4)

overfit_gap = round(train_score - test_score, 4)

print("훈련 정확도 :", train_score)
print("테스트 정확도 :", test_score)
print("과적합 차이 :", overfit_gap)

In [ ]:
dt = DecisionTreeClassifier(random_state=42,max_depth=50)
dt.fit(train_scaled, train_target)
train_score = round(dt.score(train_scaled, train_target), 4)
test_score = round(dt.score(test_scaled, test_target), 4)

overfit_gap = round(train_score - test_score, 4)

print("훈련 정확도 :", train_score)
print("테스트 정확도 :", test_score)
print("과적합 차이 :", overfit_gap)

In [ ]:
"""
<트리 가지치기 갯수 제한을 통한 튜닝>
 - 가지치기 갯수 제한 변수명(트리의 깊이를 의미함) : max_depth
 - 트리가 가지를 뻗어나가는 횟수를 제한
 - 가지가 뻗어나가는 횟수를 제한하였더라도,
   제한된 횟수 이전에 종속변수가 결정된다면 훈련을 자동 종료함
 - 종속변수가 결정되는 시점이 리프노드(leaf-node)가 결정되는 시점임
 - 리프노드가 결정되지 않으면 -> 계속 가지를 뻗어나가면서 규칙노드를 수행함
 - 이때 가지가 뻗어나가는 깊이를 트리의 깊이(max_depth)라고 칭함
   -> 트리의 깊이는 루트노드(root-node) 다음부터 깊이가 count 됨
 - 사람의 개입 시점 : 하이퍼파라메터를 통해 제어할 수 있는 값을 정의할 수는 있지만,
                    내부에서 랜덤하게 진행되는 과정은 사람이 개입하지 못함
"""

In [ ]:
best_tree = {}
for i in range(2,101) :
    dt = DecisionTreeClassifier(random_state=42,max_depth=i)
    dt.fit(train_scaled, train_target)
    train_score = round(dt.score(train_scaled, train_target), 4)
    test_score = round(dt.score(test_scaled, test_target), 4)
    overfit_gap = round(train_score - test_score, 4)
    best_tree[i] = overfit_gap
best_tree


In [ ]:
data2 = wine[["pH", "sugar"]]

train_input, test_input, train_target, test_target = train_test_split(data2, target, test_size= 0.2, stratify=target, random_state= 42)

ss.fit(train_input)
train_scaled2 = ss.transform(train_input)
test_scaled2 = ss.transform(test_input)

dt = DecisionTreeClassifier(random_state=42,max_depth=12)
dt.fit(train_scaled2, train_target)
train_score2 = round(dt.score(train_scaled2, train_target), 4)
test_score2 = round(dt.score(test_scaled2, test_target), 4)

overfit_gap2 = round(train_score2 - test_score2, 4)

print("훈련 정확도 :", train_score2)
print("테스트 정확도 :", test_score2)
print("과적합 차이 :", overfit_gap2)

#기존 훈련 : 0.95382, 0.86538, 0.08843
#새 훈련: 0.9032, 0.8408, 0.0624

In [ ]:
"""
<분류 모델 평가 방법>
 - 예측(predict)이 사전에 진행 된 후 예측결과를 이용하여 평가를 진행합니다.
 - "***정확도, 정밀도, ****재현율, **F1-Score" 값을 이용해서 평가합니다.
 - 예측(predict) 시 부정확한 오류가 있는지 확인하는 절차입니다.
 - 부정확한 오류 : 예측이 잘 못된 경우를 의미합니다.
 - 오류의 종류 : 긍정적 오류(1종오류), 부정적 오류(2종오류)로 나뉩니다.
 - 오류 확인 방법 : 오차행렬(혼동행렬)표, 오차행렬도(혼동행렬도)를 통해 확인
 
<평가 기준>
 - 정확도를 이용하여 과적합여부 확인(보통 훈련 시키는 시점에 진행, 일반화 확인을 위해)
 - 평가기준 : 일반화된 상태를 전제로 진행합니다.
   -- 주로 -> 재현율이 높고, F1-Score값이 높은 모델을 선정
   -- 재현율이 낮은 경우에는 부정적 오류(부정확한 오류)가 있다는 의미임
 - 정확도, 정밀도, 재현율, F1-Score 모든 값의 범위 : 0~1 사이의 값(확율을 의미함)
                                                  (1에 가까울 수록 좋음)
 - (주의사항)
    -- 정확도가 높더라도 재현율이 낮은 경우에는 훈련모델 사용을 고려해야함
   
 - (최종 모델 선정 기준)
    -- 일반화 > 정확도 & 재현율 > F1-Score > 정밀도
"""

In [24]:
train_pred = dt.predict(train_scaled2)
test_pred = dt.predict(test_scaled2)
confusion_matrix(train_target,train_pred,labels=dt.classes_)

array([[1105,  174],
       [ 329, 3589]])

In [29]:
train_acc = accuracy_score(train_target, train_pred)
test_acc = accuracy_score(test_target, test_pred)

train_pre = precision_score(train_target, train_pred)
test_pre = precision_score(test_target, test_pred)

train_rec = recall_score(train_target, train_pred)
test_rec = recall_score(test_target, test_pred)

train_f1 = f1_score(train_target, train_pred)
test_f1 = f1_score(test_target, test_pred)

print(f"훈련 평가결과: {train_acc:.4f}, {train_pre:.4f}, {train_rec:.4f}, {train_f1:.4f}")
print(f"테스트 평가결과: {test_acc:.4f}, {test_pre:.4f}, {test_rec:.4f}, {test_f1:.4f}")
print(f"과적합여부: {train_acc - test_acc:.4f}, {train_pre - test_pre:.4f}, {train_rec - test_rec:.4f}, {train_f1 - test_f1:.4f}")

훈련 평가결과: 0.9032, 0.9538, 0.9160, 0.9345
테스트 평가결과: 0.8408, 0.9090, 0.8765, 0.8925
과적합여부: 0.0624, 0.0448, 0.0395, 0.0420
